In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.runnables import RunnablePassthrough, RunnableParallel 
from langchain_core.output_parsers import StrOutputParser

from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""


In [19]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens=512)

In [20]:
# Carregar o PDF

pdf_link = "lei_nacionalidade_pt.pdf"
loader = PyPDFLoader(pdf_link, extract_images=False)
pages = loader.load_and_split()

In [21]:
len(pages)

15

In [22]:
# Splitters (fase chunking)

child_splitter = RecursiveCharacterTextSplitter(chunk_size=200),
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=200),
lenght_function = len,
add_start_index = True

In [23]:
# Storage

store = InMemoryStore()
vectorstore = Chroma(embedding_function=embeddings, persist_directory='childVectorDB')


In [24]:
# Fix splitter variables in case they were created as 1-item tuples
child_splitter = child_splitter[0] if isinstance(child_splitter, tuple) else child_splitter
parent_splitter = parent_splitter[0] if isinstance(parent_splitter, tuple) else parent_splitter

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

parent_document_retriever.add_documents(pages, ids=None)

In [25]:
parent_document_retriever.vectorstore.get()

{'ids': ['b63ef806-65dd-4769-a070-7f8b141d3bd0',
  'bc76984a-14d0-4719-9bb7-ba542061c126',
  'c668fbe0-09a2-4c8d-98cd-172004e78f9b',
  '5d3010f3-c338-4c94-89cc-d17c0df9763a',
  '2f7c30fa-3df7-41c9-9d29-f301eb6c73de',
  'b70d71b2-362a-4d02-8fb6-bfe4491c8931',
  'c644e252-6394-4656-9e6a-a64a7ff080e7',
  '51d25740-83b1-48dc-a90e-b37e67af3fb1',
  '10f698e0-190a-42fd-a914-e53f8354d050',
  '9f930e09-157f-4c39-a612-45d07b88c95e',
  '5be72cac-ba71-4c63-9ff5-4c4573aa392e',
  'be571481-ea9c-41fe-8597-50b5a490a300',
  '23cec91a-e67f-493c-a030-86d1a4ce90dd',
  '82625917-53a7-4a8b-af89-4dd3b933fb94',
  'a61a1d0a-591f-4322-9ca1-a7b562d1a40d',
  'f5f267de-2412-421f-9e25-0968e3dd4891',
  '3313ae9d-157e-4f09-bb7d-7e2d6afa6ad7',
  'c130da50-9541-48fe-94fb-4f166332cb92',
  'b1b07fd3-e5ad-493b-b438-e3cd1d1739ad',
  'f1c9441d-44d8-4d4a-9697-71ec0c907df1',
  '2f6db51b-3258-4bc0-b7d0-80d30ce5fb13',
  '348de810-3e97-4c65-98cc-950d3df2e92b',
  'a0010bb3-b6d8-43d7-a7b4-c01c5734e6f3',
  'c94e7ec6-c52a-4c35-8196-

In [26]:
TEMPLATE = """
    Você é um especialista em legislação portuguesa. Responda a pergunta abaixo com base no conteúdo dos documentos fornecidos. Se a resposta não estiver presente nos documentos, responda "Não sei".
    Query:
    {question}

    Context:
    {context}
"""

rag_prompt = ChatPromptTemplate.from_template(TEMPLATE)

In [28]:
setup_retrieval = RunnableParallel({"question": RunnablePassthrough(), "context": parent_document_retriever})

output_parser = StrOutputParser()

In [29]:
parent_chain_retrieval = setup_retrieval | rag_prompt | llm | output_parser

In [30]:
parent_chain_retrieval.invoke("Quem tem direito à nacionalidade portuguesa?")

'Quem tem direito à nacionalidade portuguesa são os filhos de mãe portuguesa ou de pai português nascidos no território português; os filhos de mãe portuguesa ou de pai português nascidos no estrangeiro se o progenitor português aí se encontrar ao serviço do Estado Português; os filhos de mãe portuguesa ou de pai português nascidos no estrangeiro se tiverem o seu nascimento inscrito no registo civil português ou se declararem que querem ser portugueses; os indivíduos com, pelo menos, um ascendente de nacionalidade portuguesa originária do 2.º grau na linha reta que não tenha perdido essa nacionalidade, se declararem que querem ser portugueses e possuírem laços de efetiva ligação à comunidade nacional; os indivíduos nascidos no território português, filhos de estrangeiros, se pelo menos um dos progenitores também aqui tiver nascido e aqui tiver residência, independentemente de título, ao tempo do nascimento; os indivíduos nascidos no território português, filhos de estrangeiros que não 